# 06 · Exportar para o Power BI
[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jessicapmartins/previsao-faltas-consultas/blob/main/notebooks/06_exportar_powerbi.ipynb)

Gera os arquivos do painel na pasta `dados/powerbi` do projeto. Os CSVs saem no padrão brasileiro (separador `;`, decimal com vírgula, UTF-8), para o Power BI em português ler sem ajuste.

| Arquivo | Uma linha por | Uso no painel |
|---|---|---|
| `pbi_consultas.csv` | consulta (base inteira) | visão histórica: taxa de falta por antecedência, idade, bairro, dia, SMS |
| `pbi_acoes.csv` | consulta da semana de teste | risco previsto, faixa de risco, ação recomendada e custo esperado |
| `pbi_premissas.csv` | ação | custo e efeito premissados de cada ação |
| `pbi_modelos.csv` | modelo | comparação de métricas dos modelos do notebook 03 |

In [1]:
# Funções de apoio do projeto (mesma carga, mesmas variáveis e mesmo modelo em todos os notebooks)
import os, sys, urllib.request
try:
    import google.colab
    urllib.request.urlretrieve('https://raw.githubusercontent.com/jessicapmartins/previsao-faltas-consultas/main/src/apoio.py', '/content/apoio.py')
    sys.path.insert(0, '/content')
except ModuleNotFoundError:
    sys.path.insert(0, '../src')
import apoio, pandas as pd, numpy as np

con, DADOS, CSV = apoio.conectar()

def q(sql):
    """Roda uma consulta SQL no banco de consultas e devolve um DataFrame."""
    return pd.read_sql(sql, con)

apoio.garantir(con, CSV, 'acoes')

Mounted at /content/drive


In [2]:
DIAS = {0: 'domingo', 1: 'segunda', 2: 'terça', 3: 'quarta', 4: 'quinta', 5: 'sexta', 6: 'sábado'}
base = q('SELECT * FROM base_modelo')
consultas = pd.DataFrame({
    'consulta_id':        base['consulta_id'],
    'data_consulta':      base['data_consulta'],
    'data_agendamento':   base['data_agendamento'].str[:10],
    'dia_semana':         base['dia_semana'].map(DIAS),
    'antecedencia_dias':  base['antecedencia_dias'],
    'faixa_antecedencia': pd.cut(base['antecedencia_dias'], [-1, 0, 2, 7, 14, 30, 60, 200],
                                 labels=['0 · mesmo dia', '1 · 1-2 dias', '2 · 3-7 dias', '3 · 8-14 dias',
                                         '4 · 15-30 dias', '5 · 31-60 dias', '6 · 61+ dias']).astype(str),
    'idade':              base['idade'],
    'faixa_etaria':       pd.cut(base['idade'], [-1, 12, 18, 29, 44, 59, 100],
                                 labels=['0-12', '13-18', '19-29', '30-44', '45-59', '60+']).astype(str),
    'sexo':               base['genero'].map({'F': 'Feminino', 'M': 'Masculino'}),
    'bairro':             base['bairro'].str.title(),
    'bolsa_familia':      base['bolsa_familia'], 'hipertensao': base['hipertensao'], 'diabetes': base['diabetes'],
    'alcoolismo':         base['alcoolismo'], 'deficiencia': (base['deficiencia'] > 0).astype(int),
    'sms_recebido':       base['sms_recebido'],
    'historico':          np.select([base['consultas_anteriores'] == 0, base['faltas_anteriores'] == 0],
                                    ['Primeira consulta', 'Já veio e nunca faltou'], 'Já faltou antes'),
    'faltou':             base['faltou'],
    'periodo':            np.where(base['data_consulta'] >= apoio.DATA_CORTE, 'Semana de teste', 'Histórico (treino)'),
})

acoes = q('SELECT * FROM acoes')
acoes['faixa_risco'] = pd.cut(acoes['risco'], [0, 0.1, 0.2, 0.3, 0.4, 1.0], include_lowest=True,
                              labels=['1 · até 10%', '2 · 10-20%', '3 · 20-30%', '4 · 30-40%', '5 · acima de 40%']).astype(str)
acoes['economia_esperada'] = acoes['custo_esperado_sem_acao'] - acoes['custo_esperado_acao']

premissas = apoio.ACOES.assign(custo_horario_vago=apoio.CUSTO_FALTA)
modelos = q('SELECT * FROM resultados_modelos') if 'resultados_modelos' in apoio.tabelas(con) else pd.DataFrame()
print(len(consultas), len(acoes), len(premissas), len(modelos))

110514 35240 4 0


In [3]:
PASTA = f'{DADOS}/powerbi'
os.makedirs(PASTA, exist_ok=True)
for nome, df in {'pbi_consultas': consultas, 'pbi_acoes': acoes, 'pbi_premissas': premissas, 'pbi_modelos': modelos}.items():
    if len(df):
        df.to_csv(f'{PASTA}/{nome}.csv', sep=';', decimal=',', index=False, encoding='utf-8-sig')
        print(f'{nome}.csv  {len(df):>7,} linhas · {df.shape[1]} colunas')

pbi_consultas.csv  110,514 linhas · 19 colunas
pbi_acoes.csv   35,240 linhas · 9 colunas
pbi_premissas.csv        4 linhas · 4 colunas


## Conferência rápida (os números que o painel deve mostrar)

In [4]:
print(f"Taxa de falta (base inteira): {consultas['faltou'].mean():.1%}")
print(f"Consultas na semana de teste: {len(acoes):,} · risco médio previsto {acoes['risco'].mean():.1%} · falta real {acoes['faltou'].mean():.1%}")
print(f"Economia esperada da regra na semana: R$ {acoes['economia_esperada'].sum():,.0f}")
acoes.groupby('faixa_risco').agg(consultas=('risco', 'size'), falta_real=('faltou', 'mean')).round(3)

Taxa de falta (base inteira): 20.2%
Consultas na semana de teste: 35,240 · risco médio previsto 21.6% · falta real 18.5%
Economia esperada da regra na semana: R$ 152,234


,consultas,falta_real
faixa_risco,,
1 · até 10%,11359,0.027
2 · 10-20%,4737,0.167
3 · 20-30%,8539,0.228
4 · 30-40%,6000,0.301
5 · acima de 40%,4605,0.366


**Ideias de páginas do painel:** (1) visão geral: taxa de falta, por antecedência, dia, idade e histórico; (2) mapa ou ranking de bairros; (3) agenda da semana: consultas por faixa de risco e ação recomendada, economia esperada; (4) premissas e limitações, incluindo o aviso de que o efeito do SMS não pode ser medido com esta base.

## Gravar no Google Drive
Fecha o banco e força o Drive a gravar agora. Sem isso, o próximo notebook pode não enxergar as tabelas novas.

In [5]:
con.commit()
con.close()
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print('Banco gravado no Google Drive.')
except ModuleNotFoundError:
    print('Fora do Colab: nada a sincronizar.')

Banco gravado no Google Drive.
